## MCUNet

Outputs

- `models/mcunet_quantized.tflite` INT8 input/output, per-tensor; MicroFlow-compatible

Keep `epochs` small if you only need artifacts quickly.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE_QUANT = MODELS_DIR / "mcunet_quantized.tflite"

print("OUT_TFLITE_QUANT:", OUT_TFLITE_QUANT)

In [ ]:
def conv_block(x, filters, kernel_size, strides=1):
    # MicroFlow supports fused activations in CONV_2D (NONE/RELU/RELU6).
    return tf.keras.layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)

def depthwise_separable_block(x, pointwise_filters, strides=1):
    x = tf.keras.layers.DepthwiseConv2D(
        3,
        strides=strides,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)
    x = tf.keras.layers.Conv2D(
        pointwise_filters,
        1,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)
    return x

def build_mcunet() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(32, 32, 3), batch_size=1, name="input")
    x = conv_block(inputs, 16, 3, strides=1)
    x = depthwise_separable_block(x, 32, strides=2)
    x = depthwise_separable_block(x, 64, strides=2)
    x = depthwise_separable_block(x, 64, strides=1)

    # Replace GlobalAveragePooling (-> MEAN) with AveragePooling2D (-> AVERAGE_POOL_2D).
    logits_map = tf.keras.layers.Conv2D(10, 1, padding="same", activation=None, use_bias=True)(x)
    pooled = tf.keras.layers.AveragePooling2D(pool_size=(8, 8), strides=(8, 8), padding="valid")(logits_map)
    logits = tf.keras.layers.Reshape((10,), name="logits")(pooled)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="mcunet")

model = build_mcunet()
model.summary()

In [ ]:
epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")
x_train = (x_train.astype(np.float32) / 255.0)
x_test = (x_test.astype(np.float32) / 255.0)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

In [ ]:
def representative_data_gen():
    n = min(200, x_train.shape[0])
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter.experimental_enable_resource_variables = True

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(converter, attr):
        setattr(converter, attr, False)

tflite_bytes = converter.convert()
OUT_TFLITE_QUANT.write_bytes(tflite_bytes)
print("Wrote:", OUT_TFLITE_QUANT, "bytes=", OUT_TFLITE_QUANT.stat().st_size)

interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE_QUANT), experimental_delegates=[])
interp.allocate_tensors()
in_info = interp.get_input_details()[0]
out_info = interp.get_output_details()[0]
print("int8 input:", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("int8 output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])

per_channel = []
for td in interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK")